In [7]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Data
current_dir = Path(os.getcwd())
root_dir = current_dir
for parent in [current_dir] + list(current_dir.parents):
    if (parent / "Data sets").exists():
        root_dir = parent
        break

file_path = root_dir / "Data sets" / "3) House Price Prediction.csv"
if not file_path.exists():
    data_dir = root_dir / "Data sets"
    file_path = next((f for f in data_dir.glob("*.csv") if "house" in f.name.lower() or "3)" in f.name), None)

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

target_col = next((col for col in df.columns if 'price' in col.lower() or 'target' in col.lower()), df.columns[-1])
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()

# 2. Pipeline & Hyperparameter Tuning
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid = {
    'regressor__n_estimators': [50, 100],
    'regressor__max_depth': [10, 20],
    'regressor__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_preds = best_model.predict(X_test)

tuned_rmse = np.sqrt(mean_squared_error(y_test, best_preds))
tuned_r2 = r2_score(y_test, best_preds)

print("=== TASK 3 HYPERPARAMETER OPTIMIZATION SUMMARY ===")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Optimized RMSE: ${tuned_rmse:,.2f}")
print(f"Optimized R² Score: {tuned_r2:.4f}")

=== TASK 3 HYPERPARAMETER OPTIMIZATION SUMMARY ===
Best Parameters: {'regressor__max_depth': 20, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}
Optimized RMSE: $3.93
Optimized R² Score: 0.1376


In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

# Define base pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# Define hyperparameter grid
param_grid = {
    'regressor__n_estimators': [50, 100, 150],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

# Run Grid Search Cross-Validation
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Parameters Found:")
print(grid_search.best_params_)

Best Parameters Found:
{'regressor__max_depth': 20, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}


In [6]:
# Best Model Predictions
best_model = grid_search.best_estimator_
best_preds = best_model.predict(X_test)

# Calculate metrics
tuned_rmse = np.sqrt(mean_squared_error(y_test, best_preds))
tuned_r2 = r2_score(y_test, best_preds)

optimization_results = pd.DataFrame({
    'Model Stage': ['Tuned Random Forest Regressor'],
    'Best Parameters': [str(grid_search.best_params_)],
    'Optimized RMSE': [f"${tuned_rmse:,.2f}"],
    'Optimized R² Score': [f"{tuned_r2:.4f}"]
})

print("=== TASK 3 HYPERPARAMETER OPTIMIZATION SUMMARY ===")
print(optimization_results.to_string(index=False))

=== TASK 3 HYPERPARAMETER OPTIMIZATION SUMMARY ===
                  Model Stage                                                                                 Best Parameters Optimized RMSE Optimized R² Score
Tuned Random Forest Regressor {'regressor__max_depth': 20, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}          $3.93             0.1376
